# Serve + evaluate (vLLM) — baseline (§4.4) and merged SFT (§5.5)

**Run on the L4** (serving is a throughput task — keep the A100 for training).

This notebook measures **execution accuracy on Spider `dev`** for two models with the
**identical frozen harness**, so the delta between them is meaningful:

1. **Part A — baseline:** the untrained `Qwen/Qwen2.5-Coder-7B-Instruct` (§4.4 exit criteria).
2. **Part B — SFT:** your merged 16-bit model from Phase 1 (§5.5).
3. **Part C — compare:** read the delta off `results/eval_runs.jsonl`.

The eval loop is the **frozen** `run_episode` (temperature hardcoded to 0.0, so eval is
deterministic). The *only* things that change between A and B are the served model and the
output filename — nothing else, or the comparison is invalid.


## 0. Setup


In [ ]:
!pip install -q vllm openai
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")


In [ ]:
# Repo + Drive. Edit the clone URL to your fork, then run.
import os
if not os.path.isdir("Text2SQL-Agent"):
    !git clone https://github.com/oglantz/Text2SQL-Agent.git
%cd Text2SQL-Agent

from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# Bridge code -> data. config.DATABASE_DIR resolves to <repo_root>/data/spider/...,
# but your data lives in Drive. Symlink the repo's data/ at the Drive copy (no copy,
# instant). SQLite read-only over Drive FUSE is fine for eval.
!ln -sfn /content/drive/MyDrive/Text2SQL-Agent/data data

# Verify the files eval.py needs are now reachable at the repo-relative paths.
import os
need = ["data/spider/dev.json", "data/spider/tables.json", "data/spider/database"]
missing = [p for p in need if not os.path.exists(p)]
print("Spider data missing:", missing or "none -- present")
# If FUSE ever acts up, copy the DBs to fast local disk instead of symlinking:
# !rm -f data && cp -r /content/drive/MyDrive/Text2SQL-Agent/data data

In [ ]:
# --- Config: edit these, then run everything below unchanged ---------------
import os

os.environ["DUMMY"] = "EMPTY"          # vLLM ignores the key value; the driver just needs the env var SET
BASE_URL = "http://localhost:8000/v1"
PORT = 8000

BASE_MODEL  = "Qwen/Qwen2.5-Coder-7B-Instruct"                                # §4.4 baseline
BASE_SERVED = "qwen-base"
MERGED_DIR  = "/content/drive/MyDrive/Text2SQL-Agent/weights/2000eps_1.0/qwen-sql-sft-merged"   # §5.5 your merged_16bit dir
SFT_SERVED  = "qwen-sft"

LIMIT = 0          # 0 = all of dev (~1k). Set small (e.g. 20) for a smoke test first.
OFFICIAL = False   # False = fast in-process scoring. True = official test-suite (see last section).
                   # Whatever you pick, use the SAME value for BOTH models.


## Helpers (run once)


In [ ]:
# Server lifecycle + the eval driver, so Part A and Part B are literally the same code.
import subprocess, time, gc, json, urllib.request
import torch

_server = {"proc": None, "name": None}

def start_vllm(model, served_name, port=PORT, extra=None):
    """Launch vLLM in the background, logging to vllm_<served_name>.log."""
    if _server["proc"] is not None:
        raise RuntimeError("A server is already running -- call stop_vllm() first.")
    cmd = ["python", "-m", "vllm.entrypoints.openai.api_server",
           "--model", model, "--served-model-name", served_name,
           "--enable-auto-tool-choice", "--tool-call-parser", "hermes",
           "--max-model-len", "8192", "--port", str(port)]
    if extra:
        cmd += extra
    log = open(f"vllm_{served_name}.log", "w")
    _server["proc"] = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    _server["name"] = served_name
    print(f"launching vLLM ({served_name}); tail -f vllm_{served_name}.log to watch startup")
    return _server["proc"]

def wait_ready(served_name=None, port=PORT, timeout=1800):
    """Block until the endpoint reports the model, or the process dies / times out."""
    served_name = served_name or _server["name"]
    url = f"http://localhost:{port}/v1/models"
    start = time.time()
    while time.time() - start < timeout:
        proc = _server["proc"]
        if proc is not None and proc.poll() is not None:
            raise RuntimeError(
                f"vLLM exited early (code {proc.returncode}); see vllm_{served_name}.log")
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if served_name in r.read().decode():
                    print("READY:", served_name)
                    return True
        except Exception:
            pass
        time.sleep(5)
    raise TimeoutError(f"vLLM not ready after {timeout}s; see vllm_{served_name}.log")

def stop_vllm():
    """Terminate the server and free the GPU before serving the next model."""
    proc = _server["proc"]
    if proc is None:
        print("no server running")
        return
    proc.terminate()
    try:
        proc.wait(timeout=30)
    except Exception:
        proc.kill()
    _server["proc"], _server["name"] = None, None
    gc.collect(); torch.cuda.empty_cache(); time.sleep(5)
    print("server stopped, GPU freed")

def run_eval(served_name, out_file, label, limit=None, official=None):
    """Frozen agent loop over dev -> predictions -> execution-accuracy score (logged)."""
    limit = LIMIT if limit is None else limit
    official = OFFICIAL if official is None else official
    # 1) generate predictions with the FROZEN loop (deterministic; temp is hardcoded 0.0)
    subprocess.run(
        ["python", "src/generate_trajectories.py", "--split", "dev",
         "--teacher", served_name, "--base-url", BASE_URL,
         "--api-key-env", "DUMMY", "--limit", str(limit), "--out", out_file],
        check=True)
    # 2) score and append a record to results/eval_runs.jsonl
    cmd = ["python", "src/eval.py", "--pred", out_file, "--label", label]
    if official:
        cmd.append("--official")
    subprocess.run(cmd, check=True)


## Part A — Baseline (§4.4)

The untrained student, run through the same loop you will use for the SFT model. Its EX is
the number every later phase is measured against.


In [ ]:
start_vllm(BASE_MODEL, BASE_SERVED)
wait_ready(BASE_SERVED)


In [ ]:
# OPTIONAL smoke test (20 questions) — confirm tool-calling/format work before the full run.
run_eval(BASE_SERVED, "data/trajectories/base_dev_smoke.jsonl", "base_smoke", limit=20)


In [ ]:
# Full dev baseline -> appends 'baseline_base_dev' to results/eval_runs.jsonl
run_eval(BASE_SERVED, "data/trajectories/base_dev.jsonl", "baseline_base_dev")


In [ ]:
stop_vllm()   # free the GPU + port 8000 before serving the SFT model


## Part B — Merged SFT model (§5.5)

Identical flow; only the served model and the output filename differ.


In [ ]:
import os
assert os.path.isdir(MERGED_DIR), f"merged model not found at {MERGED_DIR} -- fix MERGED_DIR in config"
start_vllm(MERGED_DIR, SFT_SERVED)
wait_ready(SFT_SERVED)


In [ ]:
# Full dev SFT eval -> appends 'sft_v1_dev' to results/eval_runs.jsonl
run_eval(SFT_SERVED, "data/trajectories/sft_dev.jsonl", "sft_v1_dev")


In [ ]:
stop_vllm()


## Part C — Compare


In [ ]:
import json
rows = [json.loads(l) for l in open("results/eval_runs.jsonl") if l.strip()]
for r in rows:
    print(f"{r['label']:>20}  EX={r['accuracy']:.3f}  "
          f"n_scored={r.get('n_scored')}  method={r['method']}")

by = {r["label"]: r for r in rows}
if "baseline_base_dev" in by and "sft_v1_dev" in by:
    d = by["sft_v1_dev"]["accuracy"] - by["baseline_base_dev"]["accuracy"]
    print(f"\nSFT - baseline delta: {d:+.3f}  ({d*100:+.1f} pts)")


**Reading the delta (§5.5):**

- **+5 to +15 pts** -> a real, demonstrable off-policy win. Persist the results and move toward Phase 2.
- **~0 or negative** -> debug in order: (1) data format / silent truncation vs `max_seq_length`;
  (2) over-training -- retry at **1 epoch**; (3) is the filter actually dropping wrong teacher
  trajectories? Do **not** advance to Phase 2 on a flat Phase 1.


In [ ]:
# results/eval_runs.jsonl lives on Colab's ephemeral disk -- persist it to Drive.
!mkdir -p /content/drive/MyDrive/Text2SQL-Agent/results
!cp results/eval_runs.jsonl /content/drive/MyDrive/Text2SQL-Agent/results/eval_runs.jsonl
print("eval_runs.jsonl copied to Drive")

## Optional — official test-suite scoring (reported numbers)

The fast in-process comparator above is fine for iterating, but it over-counts via
coincidental matches (empty sets, single values — §4.1). For the **headline** number, score
**both** models with the official Spider test-suite, which runs each query against many
perturbed DB instances and crushes the false-positive rate.

Set it up once, then set `OFFICIAL = True` in the config cell and re-run Parts A and B.
**Never compare an in-process number against an official one.**


In [ ]:
# One-time official test-suite setup.
!git clone https://github.com/taoyds/test-suite-sql-eval third_party/test-suite-sql-eval
# Download the test-suite databases (separate from Spider's normal DBs) into
# data/spider/test_suite_database/, then point eval.py at both:
import os
os.environ["SPIDER_TEST_SUITE_DIR"] = "third_party/test-suite-sql-eval"
os.environ["SPIDER_TEST_SUITE_DB"]  = "data/spider/test_suite_database"
print("official test-suite env set -- now set OFFICIAL=True and re-run Parts A and B")
